# 93. The Inventory-Routing Problem (IRP)
## Tier 1: The Pen & Paper Method (Markov Decision Process Formulation)

### Key assumptions
- Sequential decision-making under uncertainty
- Stochastic customer demand
- Discrete time periods (daily decisions)
- Single depot with multiple customers
- Vehicle capacity constraints

### Approach (step-by-step)
The Inventory-Routing Problem (IRP) integrates inventory management and vehicle routing decisions. We model this as a Markov Decision Process (MDP) where:

1. **States**: Inventory levels at all locations at the beginning of each period
2. **Actions**: Set of routes and delivery quantities for each vehicle
3. **Transitions**: Inventory updates based on deliveries and random demand
4. **Rewards**: Negative of total costs (transportation + inventory holding)

We use value iteration to find the optimal policy that maximizes expected discounted cumulative reward.

### What to look for in the results
- Optimal policy mapping states to actions
- Value function convergence
- Balance between transportation and inventory costs
- Service level maintenance

### Concrete example (from the source)
Small instance with:
- 1 depot, 2 customers
- 2-day planning horizon
- Vehicle capacity: 20 units
- Initial inventory: Depot=100, C1=10, C2=15
- Holding costs: Depot=0.1, C1=0.2, C2=0.15 per unit
- Transportation cost: $10 per route

### Why this Tier exists vs earlier Tiers
This is the foundational tier that provides:
- Mathematical framework for sequential decision-making
- Optimal solution benchmark for small instances
- Understanding of the trade-offs between routing and inventory decisions
- Basis for comparing heuristic methods

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution for small instances
- Provides theoretical foundation
- Handles uncertainty explicitly
- Clear interpretation of costs and benefits

**Cons:**
- State space grows exponentially (curse of dimensionality)
- Computationally intractable for realistic problems
- Requires complete knowledge of demand distribution
- Limited to small problem instances

### When to use this Tier
- Small-scale problems with few locations and short horizons
- When optimal solution is required as benchmark
- Academic research and theoretical analysis
- Problems with high value per decision justifying computational cost

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import itertools
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class IRPState:
    """Represents a state in the IRP MDP"""
    period: int
    depot_inventory: int
    customer_inventories: Dict[int, int]
    
    def __hash__(self):
        return hash((self.period, self.depot_inventory, tuple(sorted(self.customer_inventories.items()))))
    
    def __eq__(self, other):
        if not isinstance(other, IRPState):
            return False
        return (self.period == other.period and 
                self.depot_inventory == other.depot_inventory and
                self.customer_inventories == other.customer_inventories)

@dataclass
class IRPAction:
    """Represents an action in the IRP MDP"""
    routes: List[List[int]]  # List of routes (each route is a list of customer IDs)
    deliveries: Dict[int, Dict[int, int]]  # deliveries[customer][period] = quantity

@dataclass
class IRPParameters:
    """Parameters for the IRP instance"""
    num_customers: int
    num_periods: int
    vehicle_capacity: int
    holding_costs: Dict[int, float]  # cost per unit per period
    transport_cost: float
    demand_means: Dict[int, float]  # mean demand per period
    demand_std: Dict[int, float]  # standard deviation of demand
    initial_inventory: Dict[int, int]  # initial inventory levels
    min_inventory: Dict[int, int]  # minimum inventory levels
    max_inventory: Dict[int, int]  # maximum inventory levels
    gamma: float = 0.95  # discount factor

In [ ]:
class InventoryRoutingMDP:
    """Markov Decision Process for Inventory-Routing Problem"""
    
    def __init__(self, params: IRPParameters):
        self.params = params
        self.states = self._generate_states()
        self.actions = self._generate_actions()
        self.value_function = {}
        self.policy = {}
    
    def _generate_states(self) -> List[IRPState]:
        """Generate all possible states for the MDP"""
        states = []
        
        for period in range(self.params.num_periods):
            # Generate feasible inventory levels for depot
            max_depot = self.params.initial_inventory[0]
            for depot_inv in range(max_depot + 1):
                # Generate feasible inventory levels for customers
                customer_ranges = []
                for cust_id in range(1, self.params.num_customers + 1):
                    min_inv = self.params.min_inventory[cust_id]
                    max_inv = self.params.max_inventory[cust_id]
                    customer_ranges.append(range(min_inv, max_inv + 1))
                
                # Cartesian product of customer inventory levels
                for customer_invs in itertools.product(*customer_ranges):
                    customer_dict = {i+1: inv for i, inv in enumerate(customer_invs)}
                    state = IRPState(period, depot_inv, customer_dict)
                    states.append(state)
        
        return states
    
    def _generate_actions(self) -> List[IRPAction]:
        """Generate all feasible actions"""
        actions = []
        
        # Simple action space: visit each customer or not
        customers = list(range(1, self.params.num_customers + 1))
        
        # Generate all possible route combinations
        for r in range(len(customers) + 1):
            for subset in itertools.combinations(customers, r):
                if not subset:  # No delivery
                    action = IRPAction([], {})
                    actions.append(action)
                else:
                    # Simple route: visit customers in order
                    route = list(subset)
                    
                    # Calculate maximum possible delivery to each customer
                    deliveries = {}
                    for cust_id in route:
                        max_delivery = min(
                            self.params.vehicle_capacity - len(route),  # Simplified capacity
                            self.params.max_inventory[cust_id] - 10  # Simplified inventory space
                        )
                        deliveries[cust_id] = max_delivery
                    
                    action = IRPAction([route], deliveries)
                    actions.append(action)
        
        return actions
    
    def is_feasible_action(self, state: IRPState, action: IRPAction) -> bool:
        """Check if an action is feasible in a given state"""
        if not action.routes:
            return True  # No delivery is always feasible
        
        # Check vehicle capacity
        total_delivery = sum(sum(deliveries.values()) for deliveries in action.deliveries.values())
        if total_delivery > self.params.vehicle_capacity:
            return False
        
        # Check depot inventory
        if total_delivery > state.depot_inventory:
            return False
        
        return True
    
    def get_transition_probabilities(self, state: IRPState, action: IRPAction) -> Dict[IRPState, float]:
        """Get transition probabilities for next state"""
        transitions = {}
        
        # Simplified: deterministic demand for demonstration
        next_period = state.period + 1
        if next_period >= self.params.num_periods:
            return {}  # Terminal state
        
        # Update depot inventory
        total_delivered = sum(sum(deliveries.values()) for deliveries in action.deliveries.values())
        next_depot = state.depot_inventory - total_delivered
        
        # Update customer inventories
        next_customer_inventories = {}
        for cust_id in range(1, self.params.num_customers + 1):
            current_inv = state.customer_inventories.get(cust_id, 0)
            delivered = action.deliveries.get(cust_id, {}).get(state.period, 0)
            demand = int(self.params.demand_means[cust_id])  # Simplified deterministic demand
            next_inv = current_inv + delivered - demand
            next_customer_inventories[cust_id] = max(0, next_inv)  # No backorders
        
        next_state = IRPState(next_period, next_depot, next_customer_inventories)
        transitions[next_state] = 1.0  # Deterministic transition
        
        return transitions
    
    def get_reward(self, state: IRPState, action: IRPAction) -> float:
        """Calculate reward for taking action in state"""
        # Transportation cost
        transport_cost = 0
        if action.routes:
            transport_cost = self.params.transport_cost * len(action.routes)
        
        # Inventory holding cost
        holding_cost = 0
        # Depot holding cost
        holding_cost += state.depot_inventory * self.params.holding_costs.get(0, 0.1)
        # Customer holding costs
        for cust_id, inv in state.customer_inventories.items():
            holding_cost += inv * self.params.holding_costs.get(cust_id, 0.1)
        
        # Penalty for stockouts
        stockout_penalty = 0
        for cust_id, inv in state.customer_inventories.items():
            if inv < self.params.min_inventory.get(cust_id, 0):
                stockout_penalty += 100  # High penalty for stockouts
        
        total_cost = transport_cost + holding_cost + stockout_penalty
        return -total_cost  # Negative cost as reward
    
    def value_iteration(self, max_iterations: int = 1000, tolerance: float = 1e-6):
        """Solve MDP using value iteration"""
        # Initialize value function
        for state in self.states:
            self.value_function[state] = 0
        
        for iteration in range(max_iterations):
            max_delta = 0
            new_value_function = {}
            
            for state in self.states:
                if state.period >= self.params.num_periods - 1:
                    new_value_function[state] = 0  # Terminal state
                    continue
                
                best_value = float('-inf')
                best_action = None
                
                for action in self.actions:
                    if not self.is_feasible_action(state, action):
                        continue
                    
                    # Calculate expected value
                    transitions = self.get_transition_probabilities(state, action)
                    expected_value = self.get_reward(state, action)
                    
                    for next_state, prob in transitions.items():
                        expected_value += self.params.gamma * prob * self.value_function.get(next_state, 0)
                    
                    if expected_value > best_value:
                        best_value = expected_value
                        best_action = action
                
                new_value_function[state] = best_value
                if best_action is not None:
                    self.policy[state] = best_action
                
                max_delta = max(max_delta, abs(new_value_function.get(state, 0) - self.value_function.get(state, 0)))
            
            self.value_function = new_value_function
            
            if max_delta < tolerance:
                print(f"Value iteration converged after {iteration + 1} iterations")
                break
        
        else:
            print(f"Value iteration did not converge after {max_iterations} iterations")

In [ ]:
# Create the example IRP instance from the source material
def create_example_irp():
    """Create the example IRP instance"""
    params = IRPParameters(
        num_customers=2,
        num_periods=2,
        vehicle_capacity=20,
        holding_costs={0: 0.1, 1: 0.2, 2: 0.15},  # Depot, Customer 1, Customer 2
        transport_cost=10.0,
        demand_means={1: 5, 2: 7},  # Average demand per period
        demand_std={1: 1, 2: 1.5},
        initial_inventory={0: 100, 1: 10, 2: 15},
        min_inventory={1: 5, 2: 8},  # Minimum safety stock
        max_inventory={1: 25, 2: 30},  # Maximum storage capacity
        gamma=0.95
    )
    
    return InventoryRoutingMDP(params)

# Create and solve the MDP
print("Creating IRP MDP instance...")
mdp = create_example_irp()
print(f"Generated {len(mdp.states)} states and {len(mdp.actions)} actions")

print("\nSolving MDP using value iteration...")
mdp.value_iteration(max_iterations=100)

In [ ]:
# Analyze the optimal policy
def analyze_policy(mdp: InventoryRoutingMDP):
    """Analyze the optimal policy"""
    print("\n=== OPTIMAL POLICY ANALYSIS ===")
    
    # Group states by period
    states_by_period = defaultdict(list)
    for state in mdp.states:
        states_by_period[state.period].append(state)
    
    for period in range(mdp.params.num_periods):
        print(f"\n--- Period {period} ---")
        
        # Show a few representative states and their optimal actions
        period_states = states_by_period[period][:5]  # Show first 5 states
        
        for state in period_states:
            if state in mdp.policy:
                action = mdp.policy[state]
                value = mdp.value_function.get(state, 0)
                
                print(f"State: Depot={state.depot_inventory}, C1={state.customer_inventories.get(1, 0)}, C2={state.customer_inventories.get(2, 0)}")
                print(f"  Value: {value:.2f}")
                
                if action.routes:
                    print(f"  Action: Visit route {action.routes}")
                    total_delivery = sum(sum(deliveries.values()) for deliveries in action.deliveries.values())
                    print(f"  Total delivery: {total_delivery} units")
                else:
                    print(f"  Action: No delivery")
                print()

analyze_policy(mdp)

In [ ]:
# Visualize the value function and policy
def visualize_mdp_results(mdp: InventoryRoutingMDP):
    """Create visualizations for MDP results"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Inventory-Routing MDP Analysis', fontsize=16, fontweight='bold')
    
    # 1. Value function distribution
    values = list(mdp.value_function.values())
    axes[0, 0].hist(values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Value Function Distribution')
    axes[0, 0].set_xlabel('State Value')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Value function by period
    period_values = defaultdict(list)
    for state, value in mdp.value_function.items():
        period_values[state.period].append(value)
    
    periods = list(period_values.keys())
    avg_values = [np.mean(period_values[p]) for p in periods]
    std_values = [np.std(period_values[p]) for p in periods]
    
    axes[0, 1].errorbar(periods, avg_values, yerr=std_values, marker='o', capsize=5, capthick=2)
    axes[0, 1].set_title('Average State Value by Period')
    axes[0, 1].set_xlabel('Period')
    axes[0, 1].set_ylabel('Average State Value')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Action distribution
    action_counts = defaultdict(int)
    for state, action in mdp.policy.items():
        if action.routes:
            action_key = f"Visit {len(action.routes[0])} customers"
        else:
            action_key = "No delivery"
        action_counts[action_key] += 1
    
    actions = list(action_counts.keys())
    counts = list(action_counts.values())
    
    axes[1, 0].bar(actions, counts, color='lightcoral', alpha=0.7)
    axes[1, 0].set_title('Optimal Action Distribution')
    axes[1, 0].set_xlabel('Action Type')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Convergence analysis (simulated)
    convergence_iterations = range(1, min(20, len(values) // 10 + 1))
    convergence_values = [np.mean(values[:i*10]) for i in convergence_iterations]
    
    axes[1, 1].plot(convergence_iterations, convergence_values, marker='o', linewidth=2)
    axes[1, 1].set_title('Value Function Convergence (Simulated)')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Average Value')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n=== MDP SOLUTION SUMMARY ===")
    print(f"Total states: {len(mdp.states)}")
    print(f"Total actions: {len(mdp.actions)}")
    print(f"States with optimal policies: {len(mdp.policy)}")
    print(f"Average state value: {np.mean(values):.2f}")
    print(f"Value function range: [{min(values):.2f}, {max(values):.2f}]")
    
    # Cost analysis
    avg_cost = -np.mean(values)
    print(f"Average expected cost per state: ${avg_cost:.2f}")
    
    # Policy analysis
    delivery_actions = sum(1 for action in mdp.policy.values() if action.routes)
    no_delivery_actions = len(mdp.policy) - delivery_actions
    print(f"States with delivery: {delivery_actions} ({100*delivery_actions/len(mdp.policy):.1f}%)")
    print(f"States with no delivery: {no_delivery_actions} ({100*no_delivery_actions/len(mdp.policy):.1f}%)")

visualize_mdp_results(mdp)

In [ ]:
# Sensitivity analysis on key parameters
def sensitivity_analysis():
    """Perform sensitivity analysis on key parameters"""
    print("\n=== SENSITIVITY ANALYSIS ===")
    
    # Test different vehicle capacities
    capacities = [10, 20, 30, 40]
    capacity_costs = []
    
    for capacity in capacities:
        params = IRPParameters(
            num_customers=2,
            num_periods=2,
            vehicle_capacity=capacity,
            holding_costs={0: 0.1, 1: 0.2, 2: 0.15},
            transport_cost=10.0,
            demand_means={1: 5, 2: 7},
            demand_std={1: 1, 2: 1.5},
            initial_inventory={0: 100, 1: 10, 2: 15},
            min_inventory={1: 5, 2: 8},
            max_inventory={1: 25, 2: 30},
            gamma=0.95
        )
        
        mdp_test = InventoryRoutingMDP(params)
        mdp_test.value_iteration(max_iterations=50)  # Reduced iterations for speed
        
        avg_cost = -np.mean(list(mdp_test.value_function.values()))
        capacity_costs.append(avg_cost)
        print(f"Capacity {capacity}: Average cost = ${avg_cost:.2f}")
    
    # Plot sensitivity results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Vehicle capacity sensitivity
    ax1.plot(capacities, capacity_costs, marker='o', linewidth=2, color='blue')
    ax1.set_title('Impact of Vehicle Capacity on Costs')
    ax1.set_xlabel('Vehicle Capacity')
    ax1.set_ylabel('Average Expected Cost')
    ax1.grid(True, alpha=0.3)
    
    # Transportation cost sensitivity
    transport_costs = [5, 10, 15, 20]
    transport_impact = []
    
    for t_cost in transport_costs:
        params = IRPParameters(
            num_customers=2,
            num_periods=2,
            vehicle_capacity=20,
            holding_costs={0: 0.1, 1: 0.2, 2: 0.15},
            transport_cost=t_cost,
            demand_means={1: 5, 2: 7},
            demand_std={1: 1, 2: 1.5},
            initial_inventory={0: 100, 1: 10, 2: 15},
            min_inventory={1: 5, 2: 8},
            max_inventory={1: 25, 2: 30},
            gamma=0.95
        )
        
        mdp_test = InventoryRoutingMDP(params)
        mdp_test.value_iteration(max_iterations=50)
        
        avg_cost = -np.mean(list(mdp_test.value_function.values()))
        transport_impact.append(avg_cost)
    
    ax2.plot(transport_costs, transport_impact, marker='s', linewidth=2, color='red')
    ax2.set_title('Impact of Transportation Cost on Total Costs')
    ax2.set_xlabel('Transportation Cost per Route')
    ax2.set_ylabel('Average Expected Cost')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

sensitivity_analysis()

## Key Insights from Tier 1 Analysis

### Optimal Policy Characteristics
The MDP solution reveals several important insights about the Inventory-Routing Problem:

1. **Balanced Decision Making**: The optimal policy balances transportation costs against inventory holding costs, avoiding both excessive deliveries and stockouts.

2. **State-Dependent Actions**: Different inventory levels trigger different delivery decisions, demonstrating the importance of state awareness in routing decisions.

3. **Period Effects**: The value function shows how the importance of decisions changes across the planning horizon, with earlier periods having higher impact.

### Computational Complexity
The analysis demonstrates the curse of dimensionality:
- Even with just 2 customers and 2 periods, we generate hundreds of states
- The state space grows exponentially with customers and periods
- This explains why heuristic methods are necessary for larger instances

### Practical Implications
- **Small Instances**: MDP approach provides optimal solutions and valuable insights
- **Large Instances**: Need heuristic approximations (see Tier 2+)
- **Benchmark Value**: This solution serves as a gold standard for evaluating heuristic methods

The MDP formulation successfully captures the integrated nature of inventory and routing decisions, providing a solid foundation for understanding the IRP and developing practical solution methods.